---
title: "dict 子类的隐藏陷阱"
author: "逃之夭夭"
date: "2026-04-12"
categories: [python]
image: "images/index.jpg"
---

# UserDict vs dict

> dict 是高度优化的 C 语言实现，其内部包含一些"捷径"，使得子类化体验充满挫折且容易出 bug    ———— 《Fluent Python》

如果你只是想扩展标准字典而不影响其核心结构，直接继承 dict 完全没问题。但如果你想通过重写特殊方法来改变字典的核心行为，UserDict 才是你最好的选择。

## 一个简单示例

In [2]:
class MyDict(dict):
    def __setitem__(self, key, value):
        print(f"Setting {key} = {value}")  # 这个不会被调用
        super().__setitem__(key, value)

In [3]:
d = MyDict(a=1)

In [4]:
from collections import UserDict

class LoggingDict(UserDict):
    def __setitem__(self, key, value):
        print(f"Setting {key} = {value}")
        super().__setitem__(key, value)

In [5]:
d = LoggingDict(a=1)

Setting a = 1


## 为什么

因为 `dict` 是用 **C 语言**实现的，它的 `__init__`、`update` 等方法在 C 层直接操作底层哈希表内存，**不经过 Python 的方法查找机制[MRO](https://docs.python.org/3/howto/mro.html)**，所以你重写的 `__setitem__` 会被绕过。

而 `UserDict` 是纯 Python 实现，数据存储在 `self.data`（一个普通 dict）里，所有读写操作都通过 Python 方法调用，因此重写的方法**一定会被执行**。

## Userxxx

同理，`collections` 模块中还有以下三个 Userxxx 类，它们都避免了直接继承内置类型时的陷阱：

| 类 | 替代继承 | 数据存储 |
|---|---|---|
| `UserDict` | `dict` | `self.data` |
| `UserList` | `list` | `self.data` |
| `UserString` | `str` | `self.data`（字符串） |

三者设计哲学一致：用 `self.data` 包装底层数据，所有操作都走 Python 方法，子类可以安全重写任意方法。